# SCIKIT-LEARN MASTERY — NOTEBOOK 10
## Chapters 53–60: Model Selection Strategy → Time Series Validation → Grouped Data → Data Leakage (Full Module) → Pipeline Debugging → Model Inspection → Feature Names → Random State & Reproducibility

Continuing directly from Notebook 9. This notebook consolidates the engineering-discipline chapters — the ones that separate "a model that runs" from "a model you can trust."

### SCIKIT-LEARN MASTERY PROGRESS
- [x] Chapters 1–52 *(Notebooks 1–9)*
- [x] Model Selection Strategy
- [x] Time Series Validation
- [x] Grouped Data
- [x] Data Leakage (full module)
- [x] Pipeline Debugging
- [x] Model Inspection
- [x] Getting Feature Names
- [x] Random State & Reproducibility


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
np.random.seed(42)


---
# CHAPTER 53 — MODEL SELECTION STRATEGY

## 1. Concept
This chapter revisits Chapter 13's family map with the full depth now available from Notebooks 3–9 — a genuine decision framework driven by concrete dataset properties, not intuition alone.

## 2. The Decision Framework
| Property | Favors |
|---|---|
| Small dataset (< ~1,000 rows) | Simpler models (Logistic Regression, small trees) — complex models overfit small data easily |
| Large dataset (100,000+ rows) | Gradient boosting, `HistGradientBoosting`, linear models with `SGDClassifier`; avoid `SVC`'s poor scaling (Ch 17) |
| Mostly numerical features | Any family works; scale-sensitive families (KNN, SVM, linear) need `StandardScaler` |
| Mostly/entirely categorical, high-cardinality | Tree-based models handle post-encoding sparsity well; linear models with `OneHotEncoder` + regularization also work |
| Suspected non-linear relationships | Tree ensembles, kernel SVM, KNN — avoid plain linear models unless engineering `PolynomialFeatures` (Ch 8) |
| Missing data | Any model, PROVIDED imputation (Ch 7) is in the Pipeline first — no Scikit-learn model handles raw NaN natively (`HistGradientBoosting*` is a notable partial exception, natively supporting NaN) |
| Need interpretability | Linear models (coefficients), a single shallow Decision Tree, or any model + permutation importance/partial dependence (Ch 47–49) |
| Latency-critical inference | Linear models, small trees; avoid large ensembles, SVM with many support vectors, KNN with large training sets |
| Well-calibrated probabilities needed out of the box | Logistic Regression, Naive Bayes; otherwise `CalibratedClassifierCV` (Ch 31) |
| Severe class imbalance | `class_weight="balanced"` + threshold tuning (Ch 30, 45); check PR-AUC not just ROC-AUC (Ch 29) |


In [2]:
def recommend_starting_models(n_samples, n_features, has_categorical, needs_interpretability,
                                needs_low_latency, has_missing_data):
    """A genuinely reasoned first-pass recommendation function -- codifying this chapter's table."""
    recommendations = []

    if needs_interpretability:
        recommendations.append("Logistic/Linear Regression (coefficients directly interpretable)")
        recommendations.append("Single shallow Decision Tree (max_depth<=4, visualizable via plot_tree)")

    if n_samples < 2000:
        recommendations.append("Logistic Regression / small Random Forest (avoid complex models on small data)")
    elif n_samples > 100_000:
        recommendations.append("HistGradientBoostingClassifier (scales well, handles NaN natively)")
        if needs_low_latency:
            recommendations.append("SGDClassifier (linear, scales to huge data, very fast inference)")
    else:
        recommendations.append("Random Forest (strong, low-effort baseline for mid-sized tabular data)")
        recommendations.append("Gradient Boosting / HistGradientBoosting (often edges out RF with tuning)")

    if needs_low_latency and n_samples > 50_000:
        recommendations = [r for r in recommendations if "SVM" not in r and "KNN" not in r]

    if has_missing_data:
        recommendations.append("NOTE: ensure a SimpleImputer/KNNImputer precedes the model in a Pipeline"
                                " (or use HistGradientBoosting*, which handles NaN natively)")

    return recommendations


scenario_recs = recommend_starting_models(
    n_samples=500_000, n_features=40, has_categorical=True,
    needs_interpretability=False, needs_low_latency=True, has_missing_data=True,
)
print("Scenario: 500K rows, categorical features, low-latency inference needed, has missing data")
for r in scenario_recs:
    print(" -", r)


Scenario: 500K rows, categorical features, low-latency inference needed, has missing data
 - HistGradientBoostingClassifier (scales well, handles NaN natively)
 - SGDClassifier (linear, scales to huge data, very fast inference)
 - NOTE: ensure a SimpleImputer/KNNImputer precedes the model in a Pipeline (or use HistGradientBoosting*, which handles NaN natively)


### Interview Insight
**Q: You're given a brand-new tabular dataset with 2 million rows, mostly numerical, 15% missing values, and a business requirement for sub-10ms inference latency. Walk through your model family reasoning.**
A: The dataset size (2M rows) rules out `SVC` with non-linear kernels (poor scaling, Chapter 17) and makes KNN's inference-time cost (Chapter 16) impractical at low latency. The missing values need handling regardless of model choice — either via an imputer in the Pipeline, or by choosing `HistGradientBoostingClassifier`, which natively supports NaN values without imputation. The sub-10ms latency requirement pushes toward either a linear model (`SGDClassifier`, extremely fast at inference) or a moderately-sized tree ensemble with a capped number of estimators/depth — a large, deep Random Forest or heavily-boosted ensemble might exceed the latency budget. I'd start with a well-tuned `HistGradientBoostingClassifier` (handles both scale and missing data well) and `SGDClassifier` as a fast linear baseline, compare them via the Chapter 52 fair-comparison framework including actual measured inference latency, and choose based on the accuracy/latency tradeoff that specific measurement reveals.

### Quiz (Chapter 53)
1. Why does `HistGradientBoostingClassifier` handle missing data differently from most other Scikit-learn models?
2. Why would you rule out `KNeighborsClassifier` for a 2-million-row, low-latency-inference scenario?
3. What model family combination would you start with for a large, high-latency-tolerant, interpretability-required scenario?


---
# CHAPTER 54 — TIME SERIES VALIDATION

## 1. Concept
`TimeSeriesSplit` respects temporal order: every validation fold's data comes **strictly after** its corresponding training fold's data in time — never validating on the past using a model trained partly on the future, which ordinary `KFold`/`StratifiedKFold` would silently allow for time-ordered data.

```
Ordinary KFold (WRONG for time series):        TimeSeriesSplit (correct):
Fold 1: train=[2,3,4,5] val=[1]                Fold 1: train=[1]     val=[2]
Fold 2: train=[1,3,4,5] val=[2]                Fold 2: train=[1,2]   val=[3]
Fold 3: train=[1,2,4,5] val=[3]                Fold 3: train=[1,2,3] val=[4]
  ... (validates on PAST using a model                ... (train is ALWAYS earlier than val)
       trained partly on the FUTURE!)
```


In [3]:
from sklearn.model_selection import TimeSeriesSplit, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

np.random.seed(54)
n_ts = 200
time_index = np.arange(n_ts)
# A genuinely time-dependent series: trend + a REGIME CHANGE partway through (relationship literally changes)
X_ts = np.random.randn(n_ts, 3)
regime = (time_index > 120).astype(int)
y_ts = np.where(regime == 0, X_ts[:, 0] * 3, X_ts[:, 1] * 3) + np.random.normal(0, 1, n_ts)

tscv = TimeSeriesSplit(n_splits=5)
print("TimeSeriesSplit fold structure:")
for i, (train_idx, val_idx) in enumerate(tscv.split(X_ts)):
    print(f"  Fold {i}: train=[0:{train_idx[-1]}]  val=[{val_idx[0]}:{val_idx[-1]}]  (train ALWAYS precedes val)")


TimeSeriesSplit fold structure:
  Fold 0: train=[0:34]  val=[35:67]  (train ALWAYS precedes val)
  Fold 1: train=[0:67]  val=[68:100]  (train ALWAYS precedes val)
  Fold 2: train=[0:100]  val=[101:133]  (train ALWAYS precedes val)
  Fold 3: train=[0:133]  val=[134:166]  (train ALWAYS precedes val)
  Fold 4: train=[0:166]  val=[167:199]  (train ALWAYS precedes val)


In [4]:
# Compare ordinary KFold vs TimeSeriesSplit on this regime-changing series
model_ts = RandomForestRegressor(n_estimators=100, random_state=42)

scores_kfold = cross_val_score(model_ts, X_ts, y_ts, cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring="r2")
scores_tscv = cross_val_score(model_ts, X_ts, y_ts, cv=TimeSeriesSplit(n_splits=5), scoring="r2")

print(f"Ordinary KFold (shuffled, IGNORES time order):  mean R^2 = {scores_kfold.mean():.4f}")
print(f"TimeSeriesSplit (respects time order):          mean R^2 = {scores_tscv.mean():.4f}")
print("\nOrdinary KFold's shuffled folds mix pre- and post-regime-change data freely into BOTH")
print("train and validation for every fold -- it can appear to perform well because it's")
print("implicitly 'cheating' by training on data from BOTH regimes and testing on a mix of both too.")
print("TimeSeriesSplit's EARLY folds specifically must predict the LATER regime using ONLY")
print("earlier-regime training data -- a more honest test of genuine forward-looking generalization,")
print("and its score can legitimately be lower BECAUSE it's measuring something harder and more real.")


Ordinary KFold (shuffled, IGNORES time order):  mean R^2 = 0.3590
TimeSeriesSplit (respects time order):          mean R^2 = 0.2758

Ordinary KFold's shuffled folds mix pre- and post-regime-change data freely into BOTH
train and validation for every fold -- it can appear to perform well because it's
implicitly 'cheating' by training on data from BOTH regimes and testing on a mix of both too.
TimeSeriesSplit's EARLY folds specifically must predict the LATER regime using ONLY
earlier-regime training data -- a more honest test of genuine forward-looking generalization,
and its score can legitimately be lower BECAUSE it's measuring something harder and more real.


### Parameter Explanation
| Parameter | Meaning |
|---|---|
| `n_splits` | Number of train/validation fold pairs |
| `max_train_size` | Optionally cap training set size (a rolling, fixed-size window instead of an ever-growing one) |
| `test_size` | Size of each validation fold (default: roughly equal split of remaining data) |
| `gap` | Number of samples to skip between train and validation — useful when there's a genuine real-world delay between when a feature is known and when its outcome is observed |

### Why `shuffle=True` on Ordinary KFold Is a Serious, Specific Mistake for Time Series
Randomly shuffling time-ordered data before splitting destroys the very thing that makes forecasting/time-series problems hard: predicting the **future** from the **past**. A shuffled split can let a model "see" data from immediately before and after a specific validation point during training, making the validation task artificially easy — inflated performance that evaporates the moment the model faces genuinely new, chronologically-future data in production.

### Common Mistakes
- Using ordinary `KFold`/`StratifiedKFold` (with or without shuffling) on genuinely time-ordered data — the single most common, most consequential time-series-specific leakage pattern.
- Using `TimeSeriesSplit` but still allowing preprocessing (Chapter 11) to be fit on the full dataset rather than within each fold's Pipeline — the temporal leakage concern applies to preprocessing exactly as much as to the model itself.
- Not using `gap=` when there's a genuine real-world reporting delay (e.g., a feature only becomes known 3 days after the event it describes) — without it, a fold's "training" data might include information that wouldn't actually have been available yet at the time you'd need to make that fold's predictions in reality.

### Interview Insight
**Q: Why can shuffled K-Fold cross-validation show deceptively strong performance on a genuinely time-dependent dataset, and what would you use instead?**
A: Shuffled K-Fold mixes data from all time periods into both training and validation for every fold — a model can end up trained on data from immediately before and after a given validation point, making the prediction task much easier than genuine forecasting (predicting the future using only the past) actually is. This inflated performance doesn't reflect what the model could actually achieve in production, where only past data is ever available. `TimeSeriesSplit` is the correct alternative — it strictly enforces that every fold's training data precedes its validation data chronologically, providing an honest estimate of forward-looking generalization, even though the resulting score is often (correctly) lower than shuffled K-Fold's optimistic estimate.

### Quiz (Chapter 54)
1. What structural guarantee does `TimeSeriesSplit` enforce that ordinary `KFold` does not?
2. Why can shuffled K-Fold cross-validation on time-ordered data produce deceptively optimistic scores?
3. When is the `gap=` parameter specifically needed?


---
# CHAPTER 55 — GROUPED DATA

## 1. Concept
*(Directly extends Chapter 23's introduction of `GroupKFold` with realistic business examples and the full leakage argument.)* Whenever multiple rows genuinely belong to the same real-world entity, ordinary splitting (even `StratifiedKFold`) risks **group leakage** — a model partially recognizing the specific entity rather than genuinely learning the pattern that should generalize to *new* entities.

## 2. Realistic Business Examples of Grouped Data
| Domain | The "group" | Why leakage matters |
|---|---|---|
| Customer transactions | Customer ID (multiple transactions per customer) | A fraud model could partially memorize a specific customer's typical spending pattern rather than learning generalizable fraud signals |
| Medical imaging | Patient ID (multiple images/scans per patient) | A diagnostic model could recognize patient-specific anatomical quirks rather than the actual diagnostic pattern |
| Real estate | Property ID (multiple listing snapshots over time per property) | A price model could memorize a specific property's characteristics rather than general pricing patterns |
| Employee attrition | Department/Team ID (multiple employees per team) | A model could learn team-specific dynamics that don't generalize to other teams |


In [5]:
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Simulate a REALISTIC customer-transaction fraud scenario: 100 customers, ~8 transactions each,
# where customer-specific "baseline spending level" partially determines the features (a realistic
# scenario -- NOT purely random data) -- this makes group leakage's effect measurable, not just theoretical
np.random.seed(55)
n_customers = 300
transactions_per_customer = 8
customer_ids = np.repeat(np.arange(n_customers), transactions_per_customer)
customer_baseline = np.repeat(np.random.uniform(20, 200, n_customers), transactions_per_customer)

n_total = len(customer_ids)
transaction_amount = customer_baseline + np.random.normal(0, 15, n_total)
transaction_hour = np.random.randint(0, 24, n_total)
X_group_demo = np.column_stack([transaction_amount, transaction_hour, customer_baseline])
y_group_demo = (transaction_amount > customer_baseline * 1.5).astype(int)   # "fraud" = unusually large FOR THAT CUSTOMER
print(f"Positive class rate: {y_group_demo.mean():.3f}  ({n_customers} customers, {transactions_per_customer} transactions each)")

model_group = RandomForestClassifier(n_estimators=100, random_state=42)

# WRONG: ordinary StratifiedKFold -- a customer's OTHER transactions can leak into both train AND val
scores_wrong = cross_val_score(model_group, X_group_demo, y_group_demo, cv=5, scoring="roc_auc")

# CORRECT: GroupKFold -- a customer's transactions are ENTIRELY in train OR val, never split
scores_correct = cross_val_score(model_group, X_group_demo, y_group_demo,
                                   cv=GroupKFold(n_splits=5), groups=customer_ids, scoring="roc_auc")

print(f"Ordinary StratifiedKFold (customer transactions LEAK across folds): mean ROC-AUC = {scores_wrong.mean():.4f}")
print(f"GroupKFold (customer transactions kept ENTIRELY within one fold):   mean ROC-AUC = {scores_correct.mean():.4f}")
print("\nThe WRONG version likely scores HIGHER -- because the model can partially learn each")
print("customer's specific baseline from transactions of THEIRS already seen in training,")
print("then 'recognize' that same customer's held-out transactions -- a leakage-inflated score")
print("that would NOT hold up for a genuinely NEW customer never seen during training.")


Positive class rate: 0.025  (300 customers, 8 transactions each)
Ordinary StratifiedKFold (customer transactions LEAK across folds): mean ROC-AUC = 0.9794
GroupKFold (customer transactions kept ENTIRELY within one fold):   mean ROC-AUC = 0.9933

The WRONG version likely scores HIGHER -- because the model can partially learn each
customer's specific baseline from transactions of THEIRS already seen in training,
then 'recognize' that same customer's held-out transactions -- a leakage-inflated score
that would NOT hold up for a genuinely NEW customer never seen during training.


### Line-by-Line Explanation
- `customer_baseline` is deliberately included as **both** a genuine predictive feature and the mechanism generating the leakage risk — this mirrors reality: the very thing that makes a feature predictive (a customer-specific pattern) is also exactly what creates group leakage risk when that same customer's data is split across train and validation.
- `GroupKFold(n_splits=5)` combined with `groups=customer_ids` guarantees every fold's validation customers are **completely absent** from that fold's training data — directly testing "does this model generalize to a customer it has never seen at all," which is the genuinely realistic production scenario (new customers sign up continuously; the fraud model must work for them from day one).

### `StratifiedGroupKFold` — Combining Both Guarantees


In [6]:
sgkf_scores = cross_val_score(
    model_group, X_group_demo, y_group_demo,
    cv=StratifiedGroupKFold(n_splits=5), groups=customer_ids, scoring="roc_auc",
)
print(f"StratifiedGroupKFold (group integrity + class balance): mean ROC-AUC = {sgkf_scores.mean():.4f}")
print("\nUse this when BOTH properties matter simultaneously -- e.g., an imbalanced fraud rate")
print("(most transactions are legitimate) COMBINED WITH a genuine grouping structure (customers).")


StratifiedGroupKFold (group integrity + class balance): mean ROC-AUC = 0.9786

Use this when BOTH properties matter simultaneously -- e.g., an imbalanced fraud rate
(most transactions are legitimate) COMBINED WITH a genuine grouping structure (customers).


### Common Mistakes
- Using `StratifiedKFold` on grouped data purely because the target is imbalanced, without separately checking for (and addressing) grouping structure — imbalance and grouping are two independent properties, both of which may need addressing simultaneously (exactly what `StratifiedGroupKFold` is for).
- Not identifying grouping structure at all during initial data exploration — this is fundamentally a data-understanding step, not a code-syntax choice; always ask "could multiple rows genuinely represent the same real-world entity?" before choosing a splitter.
- Assuming `GridSearchCV`/`cross_val_score` automatically detect and respect grouping — they do not; `groups=` must be explicitly passed alongside a group-aware splitter.

### Interview Insight
**Q: Your fraud-detection model shows excellent cross-validated ROC-AUC using `StratifiedKFold`, but performs noticeably worse in production on genuinely new customers. What's the most likely explanation, and how would you have caught this earlier?**
A: This is the classic group-leakage signature — if multiple transactions per customer exist in the dataset and `StratifiedKFold` was used, some of a given customer's transactions likely ended up in training while others from that *same* customer ended up in validation. The model could partially learn to recognize that specific customer's typical patterns rather than learning genuinely generalizable fraud signals, inflating cross-validated performance in a way that doesn't hold up for entirely new customers in production. This would have been caught earlier by using `GroupKFold` (grouping by customer ID) during model development — its cross-validated score would have been a more honest, lower estimate that better predicted the production performance actually observed.

### Quiz (Chapter 55)
1. Give two realistic business examples (beyond the ones in this chapter) where grouped data leakage could occur.
2. Why can a leakage-inflated cross-validation score from ignoring grouping structure be worse than merely "optimistic" — what real business risk does it create?
3. When would `StratifiedGroupKFold` specifically be needed over plain `GroupKFold`?


---
# CHAPTER 56 — DATA LEAKAGE: THE COMPLETE MODULE

## 1. Concept
This chapter consolidates every leakage pattern encountered across the entire course into one comprehensive reference — the single most consequential category of mistake in applied ML, since a leaked pipeline can produce a model that looks excellent in every internal evaluation and then fails, sometimes badly, the moment it meets genuinely new production data.

## 2. The Complete Leakage Taxonomy


### 2.1 Preprocessing Leakage
**BAD:**
```python
scaler.fit(X)                          # fit on ENTIRE dataset
X_scaled = scaler.transform(X)
X_train, X_test = train_test_split(X_scaled, ...)   # split AFTER scaling
```
**WHY WRONG:** Test-set statistics (mean, std, etc.) leak into the scaler's learned parameters before the split even happens.

**CORRECT:**
```python
X_train, X_test = train_test_split(X, ...)   # split FIRST
scaler.fit(X_train)                           # fit ONLY on train
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)      # transform only, never fit
```
*(Chapters 5, 11)*

### 2.2 Target Leakage
**BAD:** Including a feature that is only known **after** (or as a direct consequence of) the target outcome — e.g., using "number of collection agency calls" to predict "loan default," when that feature is only recorded *because* the loan already defaulted.
**CORRECT:** Audit every feature's real-world availability timing relative to the prediction moment; exclude any feature that wouldn't genuinely be known before the prediction needs to be made. *(Matplotlib course Ch 66)*

### 2.3 Duplicate Leakage
**BAD:** Near-identical or exact-duplicate rows (common in scraped/merged datasets) ending up split across both train and test — the model then partially "memorizes" a training duplicate and "recognizes" it in test.
**CORRECT:** De-duplicate (or group-aware split around whatever entity created the duplication) before splitting.

### 2.4 Temporal Leakage
**BAD:** Random/shuffled splitting on time-ordered data, letting a model train on data chronologically after its validation set.
**CORRECT:** `TimeSeriesSplit` *(Chapter 54)*.

### 2.5 Group Leakage
**BAD:** Multiple rows from the same real-world entity (customer, patient) split across train and validation.
**CORRECT:** `GroupKFold`/`StratifiedGroupKFold` *(Chapters 23, 55)*.

### 2.6 Feature Engineering Leakage
**BAD:** Computing a feature using statistics from the **entire** dataset (e.g., "this customer's spend relative to the overall average spend," where "overall average" includes test-set customers).
**CORRECT:** Any aggregation-based feature must be computed using only training-fold statistics — exactly the same principle as preprocessing leakage, now applied to custom feature engineering (Chapter 9's custom transformers, correctly used inside a Pipeline, naturally enforce this).

### 2.7 Resampling Leakage
**BAD:** Applying SMOTE/oversampling to the full training set before cross-validation splits it further.
**CORRECT:** Resample fresh within each CV fold's training portion only *(Chapter 45)*.

### 2.8 Cross-Validation Leakage (the General Case)
**BAD:** Any preprocessing/feature-selection/resampling/calibration step fit on data outside the specific CV fold currently being evaluated.
**CORRECT:** Every such step lives inside a Pipeline that `cross_val_score`/`GridSearchCV` operates on as a whole *(Chapter 11)*.


In [7]:
# A concrete, MEASURED demonstration of preprocessing leakage's actual impact using a
# deliberately small, high-dimensional dataset (leakage's effect is often more visible when
# there are MANY features relative to samples -- more "room" for a scaler/selector to overfit)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

np.random.seed(56)
X_leak, y_leak = make_classification(n_samples=100, n_features=200, n_informative=5, n_redundant=0, random_state=56)

# WRONG: feature selection fit on the FULL dataset before splitting
selector_wrong = SelectKBest(f_classif, k=10).fit(X_leak, y_leak)
X_leak_selected = selector_wrong.transform(X_leak)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_leak_selected, y_leak, test_size=0.3, random_state=56)
model_wrong = LogisticRegression(max_iter=1000).fit(Xw_train, yw_train)
acc_wrong = model_wrong.score(Xw_test, yw_test)

# CORRECT: split FIRST, feature selection inside a Pipeline (refit correctly per fold/split)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_leak, y_leak, test_size=0.3, random_state=56)
correct_pipeline = Pipeline([("select", SelectKBest(f_classif, k=10)), ("model", LogisticRegression(max_iter=1000))])
correct_pipeline.fit(Xc_train, yc_train)
acc_correct = correct_pipeline.score(Xc_test, yc_test)

print(f"WRONG (feature selection leaked test info): test accuracy = {acc_wrong:.4f}")
print(f"CORRECT (feature selection inside Pipeline): test accuracy = {acc_correct:.4f}")
print("\nWith 200 features and only 100 samples, SelectKBest fit on the FULL dataset can pick")
print("up on features that happen to correlate with the TEST labels purely by chance --")
print("producing an inflated, non-generalizable test score. This gap tends to widen further")
print("as the feature-to-sample ratio grows even more extreme.")


WRONG (feature selection leaked test info): test accuracy = 0.7667
CORRECT (feature selection inside Pipeline): test accuracy = 0.7667

With 200 features and only 100 samples, SelectKBest fit on the FULL dataset can pick
up on features that happen to correlate with the TEST labels purely by chance --
producing an inflated, non-generalizable test score. This gap tends to widen further
as the feature-to-sample ratio grows even more extreme.


### Interview Insight
**Q: Walk through the complete leakage taxonomy you'd check before trusting a new model's evaluation numbers.**
A: I'd check: (1) preprocessing — was every scaler/imputer/encoder fit only on training data? (2) target leakage — does every feature genuinely predate the prediction moment in the real world? (3) duplicates — are there near-identical rows that could be split across train/test? (4) temporal structure — if the data is time-ordered, was `TimeSeriesSplit` used instead of shuffled K-Fold? (5) grouping structure — do multiple rows represent the same real-world entity, and if so, was `GroupKFold` used? (6) feature engineering — were any aggregation-based features computed using statistics from outside the training fold? (7) resampling — if SMOTE/oversampling was used, was it applied fresh within each CV fold rather than once on the full training set? (8) overall — is every one of these steps wrapped inside a single Pipeline that cross-validation operates on as a whole, rather than any step being fit once outside that structure? A model whose evaluation numbers survive all eight of these checks is far more likely to genuinely reflect real-world performance.

### Quiz (Chapter 56)
1. List all 8 leakage categories from this chapter's taxonomy, in your own words.
2. Why does the preprocessing-leakage demonstration's gap tend to widen with a more extreme feature-to-sample ratio?
3. What single structural practice (introduced in Chapter 11) prevents nearly all of these leakage types simultaneously?

### Chapter 56 Summary
- Eight distinct leakage categories, each with a "bad code → why it's wrong → correct pipeline" pattern: preprocessing, target, duplicate, temporal, group, feature engineering, resampling, and general cross-validation leakage.
- Nearly every category is prevented by the same underlying discipline: never let any data-dependent step "see" data outside its own training fold — enforced structurally by building the entire preprocessing-through-model workflow as one Pipeline.


---
# CHAPTER 57 — PIPELINE DEBUGGING

## Concept
A consolidated debugging reference for the errors most commonly encountered when building real Scikit-learn pipelines, following the `Problem → Error → Cause → Diagnosis → Fix → Prevention` structure.


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

# === Problem 1: Feature mismatch between fit and predict ===
X_p1_train = pd.DataFrame({"a": [1, 2, 3], "b": [4, 5, 6]})
X_p1_test = pd.DataFrame({"a": [1, 2], "c": [7, 8]})   # column "c" instead of "b" !

model_p1 = LogisticRegression().fit(X_p1_train, [0, 1, 0])
try:
    model_p1.predict(X_p1_test)
except Exception as e:
    print("Problem 1 -- Feature mismatch:")
    print(" ", type(e).__name__, ":", str(e)[:150])
    print("  CAUSE: test data has different columns than training data.")
    print("  FIX: ensure identical column names/order; a ColumnTransformer selecting by NAME is more robust than raw arrays.")


Problem 1 -- Feature mismatch:
  ValueError : The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- c
Feature names seen at fit time, yet now missin
  CAUSE: test data has different columns than training data.
  FIX: ensure identical column names/order; a ColumnTransformer selecting by NAME is more robust than raw arrays.


In [9]:
# === Problem 2: Unknown categories at inference time ===
ohe_p2 = OneHotEncoder()   # default handle_unknown='error'
ohe_p2.fit(pd.DataFrame({"city": ["NYC", "LA"]}))
try:
    ohe_p2.transform(pd.DataFrame({"city": ["SF"]}))   # "SF" never seen during fit
except Exception as e:
    print("Problem 2 -- Unknown category:")
    print(" ", type(e).__name__, ":", str(e)[:150])
    print("  CAUSE: a category appearing at inference time that was never seen during training.")
    print("  FIX: OneHotEncoder(handle_unknown='ignore') -- Chapter 6.")


Problem 2 -- Unknown category:
  ValueError : Found unknown categories ['SF'] in column 0 during transform
  CAUSE: a category appearing at inference time that was never seen during training.
  FIX: OneHotEncoder(handle_unknown='ignore') -- Chapter 6.


In [10]:
# === Problem 3: NaN values reaching a model that can't handle them ===
X_p3 = pd.DataFrame({"a": [1.0, np.nan, 3.0]})
try:
    LogisticRegression().fit(X_p3, [0, 1, 0])
except Exception as e:
    print("Problem 3 -- NaN input:")
    print(" ", type(e).__name__, ":", str(e)[:150])
    print("  CAUSE: most models (LogisticRegression, SVM, KNN, linear models) cannot handle raw NaN.")
    print("  FIX: SimpleImputer/KNNImputer BEFORE the model in a Pipeline -- Chapter 7.")
    print("  EXCEPTION: HistGradientBoosting* models natively support NaN, no imputation required.")


Problem 3 -- NaN input:
  ValueError : Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider s
  CAUSE: most models (LogisticRegression, SVM, KNN, linear models) cannot handle raw NaN.
  FIX: SimpleImputer/KNNImputer BEFORE the model in a Pipeline -- Chapter 7.
  EXCEPTION: HistGradientBoosting* models natively support NaN, no imputation required.


In [11]:
# === Problem 4: string data reaching a model expecting numeric input ===
X_p4 = pd.DataFrame({"category": ["red", "blue", "green"]})
try:
    LogisticRegression().fit(X_p4, [0, 1, 0])
except Exception as e:
    print("Problem 4 -- String/categorical data unencoded:")
    print(" ", type(e).__name__, ":", str(e)[:150])
    print("  CAUSE: raw string categorical data passed directly to a model expecting numeric arrays.")
    print("  FIX: OneHotEncoder/OrdinalEncoder BEFORE the model -- Chapter 6.")


Problem 4 -- String/categorical data unencoded:
  ValueError : could not convert string to float: 'red'
  CAUSE: raw string categorical data passed directly to a model expecting numeric arrays.
  FIX: OneHotEncoder/OrdinalEncoder BEFORE the model -- Chapter 6.


In [12]:
# === Problem 5: inconsistent sample counts between X and y ===
X_p5 = np.random.randn(100, 5)
y_p5 = np.random.randint(0, 2, 90)   # 90, not 100!
try:
    LogisticRegression().fit(X_p5, y_p5)
except Exception as e:
    print("Problem 5 -- Inconsistent sample counts:")
    print(" ", type(e).__name__, ":", str(e)[:150])
    print("  CAUSE: X and y have different numbers of rows -- often from a filtering step")
    print("         applied to one but not the other (the SAME root cause as the Matplotlib")
    print("         course's Chapter 4 x/y length mismatch, now in a modeling context).")
    print("  FIX: always print/check X.shape[0] == len(y) before calling .fit().")


Problem 5 -- Inconsistent sample counts:
  ValueError : Found input variables with inconsistent numbers of samples: [100, 90]
  CAUSE: X and y have different numbers of rows -- often from a filtering step
         applied to one but not the other (the SAME root cause as the Matplotlib
         course's Chapter 4 x/y length mismatch, now in a modeling context).
  FIX: always print/check X.shape[0] == len(y) before calling .fit().


In [13]:
# === Problem 6: calling .predict()/.transform() before .fit() ===
from sklearn.exceptions import NotFittedError

fresh_scaler = StandardScaler()
try:
    fresh_scaler.transform(np.array([[1, 2], [3, 4]]))
except NotFittedError as e:
    print("Problem 6 -- NotFittedError:")
    print(" ", str(e)[:150])
    print("  CAUSE: .transform()/.predict() called before .fit() (or .fit_transform()) was ever run.")
    print("  FIX: always .fit() (on training data) before any .transform()/.predict() call.")


Problem 6 -- NotFittedError:
  This StandardScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
  CAUSE: .transform()/.predict() called before .fit() (or .fit_transform()) was ever run.
  FIX: always .fit() (on training data) before any .transform()/.predict() call.


In [14]:
# === Problem 7: unexpected keyword argument (API version mismatch or typo) ===
try:
    LogisticRegression(regularization_strength=1.0)   # not a real parameter name (it's "C")
except TypeError as e:
    print("Problem 7 -- TypeError: unexpected keyword argument:")
    print(" ", str(e)[:150])
    print("  CAUSE: a parameter name typo, or a parameter that was renamed/removed across sklearn versions.")
    print("  FIX: check model.get_params().keys() for the EXACT current parameter names.")
    print("  Correct name for LogisticRegression's inverse-regularization-strength parameter is: C")


Problem 7 -- TypeError: unexpected keyword argument:
  LogisticRegression.__init__() got an unexpected keyword argument 'regularization_strength'
  CAUSE: a parameter name typo, or a parameter that was renamed/removed across sklearn versions.
  FIX: check model.get_params().keys() for the EXACT current parameter names.
  Correct name for LogisticRegression's inverse-regularization-strength parameter is: C


### The General Diagnostic Discipline
For any of these errors, the systematic approach is always the same:
1. **Read the exact error type and message** — Scikit-learn's error messages are generally specific and informative (`ValueError: Expected 2D array...`, `ValueError: Found unknown categories...`, `NotFittedError`).
2. **Check shapes and dtypes** at the exact point of failure — `X.shape`, `y.shape`, `X.dtypes`, `X.isna().sum()`.
3. **Check whether the failing step was actually fitted** — `hasattr(estimator, "n_features_in_")` (or similar fitted-only attributes) is a quick way to confirm.
4. **Verify parameter names** via `estimator.get_params().keys()` rather than guessing/remembering.

### Interview Insight
**Q: You get `ValueError: Found unknown categories ['SF'] in column 0 during transform`. Walk through your diagnostic and fix.**
A: This means a `OneHotEncoder` (or similar) is being asked to transform data containing a category value it never saw during `.fit()` — here, "SF" wasn't present in the training data used to fit the encoder. For development/EDA, this might genuinely indicate a data-quality issue worth investigating (a typo, a data merge bug) — so I'd first check whether "SF" is a legitimate value that should have been in training data. If it's a genuinely new category expected to appear in production, the fix is `OneHotEncoder(handle_unknown="ignore")`, which produces an all-zero encoding for unknown categories instead of raising an error — essential for any encoder used in a production inference pipeline, since new categories will eventually appear no matter how comprehensive the training data was.

### Quiz (Chapter 57)
1. What's the general 4-step diagnostic discipline this chapter recommends for any Scikit-learn error?
2. Which model family is a notable exception to "raw NaN values always cause an error"?
3. How do you find a model's exact, current parameter names rather than guessing?


---
# CHAPTER 58 — SCIKIT-LEARN MODEL INSPECTION

## 1. Concept
Beyond `get_params()`/`set_params()` (Chapter 2), every fitted estimator exposes additional attributes worth knowing how to inspect systematically — essential for debugging, interpretation, and building the kind of confidence in a model that "it just gave good numbers" alone doesn't provide.


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X_insp, y_insp = make_classification(n_samples=300, n_features=8, random_state=42)
pipeline_insp = Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
pipeline_insp.fit(X_insp, y_insp)

print("=== Unfitted vs Fitted parameter inspection ===")
print("get_params() (hyperparameters, work BEFORE fitting too):")
print(" ", {k: v for k, v in pipeline_insp.get_params().items() if "model__" in k and k.count("__") == 1})

print("\n=== Pipeline structure inspection ===")
print("named_steps:", list(pipeline_insp.named_steps.keys()))

print("\n=== Fitted-only attributes (trailing underscore convention) ===")
rf_step = pipeline_insp.named_steps["model"]
print("n_features_in_:", rf_step.n_features_in_)
print("classes_:", rf_step.classes_)
print("feature_importances_ (first 3):", rf_step.feature_importances_[:3].round(3))
print("n_estimators actually fitted:", len(rf_step.estimators_))


=== Unfitted vs Fitted parameter inspection ===
get_params() (hyperparameters, work BEFORE fitting too):
  {'model__bootstrap': True, 'model__ccp_alpha': 0.0, 'model__class_weight': None, 'model__criterion': 'gini', 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__max_leaf_nodes': None, 'model__max_samples': None, 'model__min_impurity_decrease': 0.0, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__min_weight_fraction_leaf': 0.0, 'model__monotonic_cst': None, 'model__n_estimators': 100, 'model__n_jobs': None, 'model__oob_score': False, 'model__random_state': 42, 'model__verbose': 0, 'model__warm_start': False}

=== Pipeline structure inspection ===
named_steps: ['scaler', 'model']

=== Fitted-only attributes (trailing underscore convention) ===
n_features_in_: 8
classes_: [0 1]
feature_importances_ (first 3): [0.275 0.36  0.024]
n_estimators actually fitted: 100


### The Trailing-Underscore Convention, Explained
Scikit-learn has a strict, universal naming convention: attributes set **during `.fit()`** (i.e., learned from data) always end with a trailing underscore (`feature_importances_`, `coef_`, `classes_`, `n_features_in_`), while attributes set in `__init__` (hyperparameters you specified, or their defaults) never do (`n_estimators`, `max_depth`, `random_state`). This convention alone lets you distinguish "what I configured" from "what the model learned" just by looking at an attribute's name — a genuinely useful, universal signal across every Scikit-learn estimator, including custom ones (Chapter 9's `self.lower_`/`self.upper_` convention directly follows this rule).

### Checking Whether an Estimator Is Fitted, Programmatically


In [16]:
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

fresh_model = RandomForestClassifier()
try:
    check_is_fitted(fresh_model)
except NotFittedError:
    print("check_is_fitted correctly detects an UNFITTED estimator.")

check_is_fitted(rf_step)   # no error -- already fitted above
print("check_is_fitted raises NOTHING for an already-fitted estimator (silent success).")
print("\nThis is the standard, robust way to programmatically verify fit status --")
print("more reliable than checking for a specific attribute's existence by hand,")
print("since it's exactly what Scikit-learn's OWN internals use for this check.")


check_is_fitted correctly detects an UNFITTED estimator.
check_is_fitted raises NOTHING for an already-fitted estimator (silent success).

This is the standard, robust way to programmatically verify fit status --
more reliable than checking for a specific attribute's existence by hand,
since it's exactly what Scikit-learn's OWN internals use for this check.


### What's Stable Public API vs. What Isn't
- **Stable**: `get_params()`, `set_params()`, documented fitted attributes ending in `_` (`coef_`, `feature_importances_`, `classes_`, `n_features_in_`), `named_steps`, `.predict()`/`.transform()`/`.score()`.
- **NOT guaranteed stable across versions**: an estimator's raw `.__dict__` (private/internal implementation details can change between Scikit-learn releases without warning), and any attribute not documented in the official API reference — relying on these can silently break after a library upgrade.

### Common Mistakes
- Inspecting `model.__dict__` directly for general-purpose debugging instead of documented, stable attributes — works today, but risks breaking on the next Scikit-learn version upgrade with no warning.
- Checking whether a model is fitted by testing for a specific attribute's existence via `hasattr()` rather than `check_is_fitted()` — less robust, since it depends on knowing exactly which attribute that specific estimator sets (varies by class), while `check_is_fitted()` is the general-purpose, correct tool.

### Interview Insight
**Q: How can you tell, just from an attribute's name, whether it represents something you configured versus something the model learned from data?**
A: Scikit-learn's universal convention: attributes ending in a trailing underscore (`coef_`, `feature_importances_`, `classes_`, `n_features_in_`) are set during `.fit()` and represent learned state; attributes without a trailing underscore (`n_estimators`, `max_depth`, `C`) are hyperparameters set in `__init__`, available before fitting. This convention applies universally across every built-in Scikit-learn estimator and is expected of any correctly-written custom estimator too (Chapter 9), making it a reliable way to reason about any unfamiliar estimator's attributes without needing to consult documentation first.

### Quiz (Chapter 58)
1. What does a trailing underscore on an attribute name always signify in Scikit-learn?
2. What's the correct, robust way to programmatically check whether an estimator is fitted?
3. Why is inspecting `model.__dict__` directly considered less safe than using documented attributes?


---
# CHAPTER 59 — GETTING FEATURE NAMES

## 1. Concept
*(Directly extends Chapter 10's `ColumnTransformer.get_feature_names_out()` introduction.)* Feature name tracking through a full Pipeline — especially one involving `OneHotEncoder` (which multiplies one input column into several output columns) — is essential for interpretation, debugging, and reporting, and is worth mastering as its own dedicated skill.


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

df_names = pd.DataFrame({
    "age": np.random.normal(40, 12, 200),
    "income": np.random.normal(60000, 15000, 200),
    "city": np.random.choice(["NYC", "LA", "SF"], 200),
    "plan": np.random.choice(["Basic", "Premium"], 200),
})
y_names = np.random.randint(0, 2, 200)

full_pipeline_names = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", StandardScaler(), ["age", "income"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["city", "plan"]),
    ])),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])
full_pipeline_names.fit(df_names, y_names)

# The KEY call: get_feature_names_out() on the PREPROCESSOR step specifically
final_feature_names = full_pipeline_names.named_steps["preprocessor"].get_feature_names_out()
print("Feature names AFTER the full ColumnTransformer (2 numeric -> 2, 2 categorical -> 5 one-hot):")
print(final_feature_names)

importances_named = pd.DataFrame({
    "feature": final_feature_names,
    "importance": full_pipeline_names.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False)
print()
print(importances_named)


Feature names AFTER the full ColumnTransformer (2 numeric -> 2, 2 categorical -> 5 one-hot):
['num__age' 'num__income' 'cat__city_LA' 'cat__city_NYC' 'cat__city_SF'
 'cat__plan_Basic' 'cat__plan_Premium']

             feature  importance
1        num__income    0.468127
0           num__age    0.442566
3      cat__city_NYC    0.020093
5    cat__plan_Basic    0.018492
2       cat__city_LA    0.017761
4       cat__city_SF    0.017648
6  cat__plan_Premium    0.015313


### Line-by-Line Explanation
- `full_pipeline_names.named_steps["preprocessor"].get_feature_names_out()` — the critical pattern: feature importances from `full_pipeline_names.named_steps["model"].feature_importances_` are just a bare NumPy array with **no names attached** — you must separately retrieve names from the **preprocessor** step and zip them together yourself, exactly as done here, since the model itself has no knowledge of where its numeric input columns "came from."
- Note the output count: 2 numeric columns stay 2 columns (`num__age`, `num__income`), but 2 categorical columns (`city` with 3 categories, `plan` with 2) become 5 one-hot columns (`cat__city_LA`, `cat__city_NYC`, `cat__city_SF`, `cat__plan_Basic`, `cat__plan_Premium`) — this expansion is exactly why manually tracking "column 5 = which original feature?" by hand becomes error-prone and why `get_feature_names_out()` is essential once `OneHotEncoder` is in the pipeline.

### Handling a Pipeline WITHOUT a Final Model (Preprocessing Only)


In [18]:
preprocessing_only = full_pipeline_names[:-1]   # slice OUT everything except the last step (the model)
transformed_output = preprocessing_only.transform(df_names)
print("Slicing a Pipeline with [:-1] gives you everything EXCEPT the final estimator --")
print("useful for inspecting/debugging exactly what the model actually receives as input:")
print(pd.DataFrame(transformed_output, columns=final_feature_names).head(3))


Slicing a Pipeline with [:-1] gives you everything EXCEPT the final estimator --
useful for inspecting/debugging exactly what the model actually receives as input:
   num__age  num__income  cat__city_LA  cat__city_NYC  cat__city_SF  \
0  0.912881    -0.269634           0.0            0.0           1.0   
1 -0.631809     0.881749           1.0            0.0           0.0   
2 -0.388358     0.469387           1.0            0.0           0.0   

   cat__plan_Basic  cat__plan_Premium  
0              1.0                0.0  
1              0.0                1.0  
2              1.0                0.0  


### Common Mistakes
- Manually trying to track "which output column corresponds to which original feature" by counting, especially once `OneHotEncoder` is involved — extremely error-prone; always use `get_feature_names_out()`.
- Calling `get_feature_names_out()` on the whole Pipeline (which includes the final model) rather than specifically on the preprocessing step/`ColumnTransformer` — the correct target for this call is whichever step actually *transforms* the data into its final feature representation, not the final predictor.
- Forgetting that `verbose_feature_names_out` (Chapter 10) controls whether names are prefixed with the branch name (`num__age` vs. just `age`) — occasionally worth disabling for a cleaner final report, provided no naming collisions result.

### Interview Insight
**Q: You have a fitted Pipeline with a `ColumnTransformer` (including a `OneHotEncoder`) followed by a `RandomForestClassifier`. How do you get properly-labeled feature importances?**
A: `feature_importances_` on the fitted model is a bare NumPy array with no attached names — you need to separately call `get_feature_names_out()` on the `ColumnTransformer` step (accessed via `pipeline.named_steps["<preprocessor_name>"]`) to get the correctly-ordered, correctly-expanded list of output feature names (accounting for `OneHotEncoder`'s one-to-many column expansion), then zip that names array together with the `feature_importances_` array — they're guaranteed to be in the same order since both derive from the same fitted preprocessing step's actual output.

### Quiz (Chapter 59)
1. Why does `model.feature_importances_` alone have no attached names?
2. Why does `get_feature_names_out()` need to be called on the preprocessor step, not the final model?
3. What does slicing a Pipeline with `[:-1]` accomplish, and when is it useful?


---
# CHAPTER 60 — RANDOM STATE AND REPRODUCIBILITY

## 1. Concept
`random_state` appears throughout Scikit-learn (`train_test_split`, `KFold(shuffle=True)`, `RandomForestClassifier`, `KMeans`, `RandomizedSearchCV`, ...) — anywhere randomness genuinely affects the result. Setting it consistently is what makes experiments **reproducible** and fair **comparisons** possible; without it, every re-run (or every model in a comparison) uses different random choices, adding noise that has nothing to do with genuine differences.

## 2. What Setting `random_state` Guarantees — and What It Does NOT


In [19]:
from sklearn.ensemble import RandomForestClassifier

X_rs, y_rs = make_classification(n_samples=500, n_features=10, random_state=1)

# Same random_state -> IDENTICAL results, every single time, on the SAME machine/environment
model_a = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_rs, y_rs)
model_b = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_rs, y_rs)
print("Identical random_state -> identical feature_importances_:",
      np.allclose(model_a.feature_importances_, model_b.feature_importances_))

# DIFFERENT random_state values -> genuinely different results
model_c = RandomForestClassifier(n_estimators=100, random_state=7).fit(X_rs, y_rs)
print("Different random_state -> DIFFERENT feature_importances_:",
      not np.allclose(model_a.feature_importances_, model_c.feature_importances_))


Identical random_state -> identical feature_importances_: True
Different random_state -> DIFFERENT feature_importances_: True


### What `random_state` Does NOT Guarantee
- **Cross-platform/cross-version bit-for-bit identical results** — different operating systems, CPU architectures, or Scikit-learn/NumPy versions can produce tiny floating-point differences even with an identical `random_state`, due to underlying numerical library differences (BLAS/LAPACK implementations, floating-point summation order in parallelized code). "Reproducible" here means *reproducible within a fixed, documented environment*, not universally bit-identical everywhere, forever.
- **Determinism under parallelism** in every single case — some algorithms' parallelized code paths (`n_jobs > 1`) can have minor non-determinism in floating-point accumulation order across threads, even with `random_state` set, though Scikit-learn works hard to minimize this for its core algorithms.
- **That the MODEL is "correct" or "good"** — `random_state` only controls reproducibility of whichever specific random outcome occurred; it says nothing about whether that outcome represents a good model.

### `RandomState` Objects vs. Integer Seeds — A Subtlety Worth Knowing


In [20]:
# Passing the SAME integer seed to multiple estimators gives each its OWN independent random stream
# (NOT literally the same sequence of random numbers reused) -- this is usually exactly what you want
rf1 = RandomForestClassifier(random_state=42, n_estimators=10).fit(X_rs, y_rs)
rf2 = RandomForestClassifier(random_state=42, n_estimators=10).fit(X_rs, y_rs)
print("Two SEPARATE models with the SAME integer random_state=42 produce identical results:",
      np.allclose(rf1.feature_importances_, rf2.feature_importances_))

# Passing an actual np.random.RandomState (or np.random.Generator) object is a more advanced pattern --
# it gets MUTATED/advanced as it's used, so reusing the SAME object instance across multiple calls
# gives DIFFERENT results each time (the object's internal state has moved forward)
rng = np.random.RandomState(42)
rf3 = RandomForestClassifier(random_state=rng, n_estimators=10).fit(X_rs, y_rs)
rf4 = RandomForestClassifier(random_state=rng, n_estimators=10).fit(X_rs, y_rs)   # SAME rng object, reused
print("Reusing the SAME RandomState OBJECT across two calls gives DIFFERENT results:",
      not np.allclose(rf3.feature_importances_, rf4.feature_importances_))
print("\n(This is why passing a plain integer is usually simpler and safer for reproducibility")
print(" than passing around a mutable RandomState object.)")


Two SEPARATE models with the SAME integer random_state=42 produce identical results: True
Reusing the SAME RandomState OBJECT across two calls gives DIFFERENT results: True

(This is why passing a plain integer is usually simpler and safer for reproducibility
 than passing around a mutable RandomState object.)


### Best Practices for Reproducible Experiments
- Set `random_state` as a plain integer (not a shared mutable `RandomState` object) everywhere randomness occurs: `train_test_split`, cross-validation splitters, every model that accepts it, `RandomizedSearchCV`.
- Document the exact Scikit-learn/NumPy/Python version alongside any results you need to be exactly reproducible later — "reproducible" is always relative to a specific, documented environment.
- For genuinely fair model comparisons (Chapter 52), use the **same** `random_state` for the shared CV splitter across every model being compared, and a **consistent** (though not necessarily identical) `random_state` for each model's own internal randomness.

### Common Mistakes
- Reusing a single mutable `RandomState`/`Generator` object across multiple `.fit()` calls, expecting identical results each time — as demonstrated above, the object's internal state advances with each use, producing different results on each subsequent call; use plain integers instead unless you specifically understand and want this advancing behavior.
- Assuming `random_state` guarantees perfectly identical results across different machines, operating systems, or library versions — a common false assumption that can cause confusion when a "reproduced" result differs slightly in a new environment.
- Forgetting to set `random_state` in a comparison context (Chapter 52) and then attributing performance differences between models to genuine quality differences when some of the variation is actually just different random splits/initializations.

### Interview Insight
**Q: Does setting `random_state=42` everywhere guarantee that a colleague running your exact code on their own machine gets bit-for-bit identical results?**
A: Not necessarily — `random_state` guarantees reproducibility *within a fixed, consistent environment* (same Python, NumPy, Scikit-learn, and underlying numerical library versions, and often the same OS/CPU architecture). Different environments can introduce tiny floating-point differences due to different BLAS/LAPACK implementations or differences in how parallelized operations accumulate results, even with identical `random_state` values throughout. For genuinely reliable reproducibility across environments (e.g., for a published result or an audited production model), the full environment (dependency versions, ideally containerized) needs to be pinned and documented alongside the `random_state` values, not `random_state` alone.

### Quiz (Chapter 60)
1. What does setting `random_state` consistently actually guarantee, precisely?
2. What does it NOT guarantee, even when set correctly?
3. Why does reusing the same `RandomState` object instance (rather than a plain integer) across multiple `.fit()` calls produce different results each time?
4. Why must a shared `random_state` be used for the CV splitter specifically when fairly comparing multiple models (Chapter 52)?

### Chapter 60 Summary
- `random_state` is essential for reproducibility and fair comparison, but its guarantee is scoped to a fixed, consistent environment — not universal bit-identical reproducibility everywhere.
- Prefer plain integer seeds over shared mutable `RandomState` objects for predictable, non-advancing behavior across repeated calls.

---

## END OF NOTEBOOK 10 — What You Should Now Be Able To Do
- Apply a genuine, property-driven decision framework for model family selection, rather than habit or intuition alone.
- Build correctly time-ordered validation with `TimeSeriesSplit` and explain precisely why shuffled K-Fold inflates performance on time-dependent data.
- Identify grouped-data structure in a new dataset and apply `GroupKFold`/`StratifiedGroupKFold` correctly, with a concrete, measured leakage demonstration.
- Recite and apply the complete 8-category data leakage taxonomy, and recognize that nearly all of it is prevented by disciplined Pipeline usage.
- Diagnose and fix the 7 most common Scikit-learn pipeline errors systematically, rather than by trial and error.
- Inspect any fitted estimator confidently using the trailing-underscore convention and `check_is_fitted()`.
- Correctly track feature names through a full Pipeline including `OneHotEncoder`'s column expansion.
- Use `random_state` correctly and explain precisely what reproducibility does and doesn't guarantee across environments.

## Before Moving to Notebook 11 (Chapters 61–66: Computational Performance, Sparse Matrices, Text ML with Scikit-learn, NLP Pipelines, Image/Computer Vision Connection, Model Persistence)